In [ ]:
# ==============================================================================
# Cell 0: Setup - Download FSC Dataset + Clone Repo
# ==============================================================================
import os, subprocess
subprocess.run(['pip', 'install', '-q', 'soundfile'], check=True)

REPO = '/content/NC-KWS-FineTuning'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/DrJinHoChoi/NC-KWS-FineTuning.git', REPO], check=True)
os.chdir(REPO)

FSC_DIR = '/content/fluent_speech_commands_dataset'
if not os.path.exists(FSC_DIR):
    print('Downloading Fluent Speech Commands (~1.5GB)...')
    try:
        subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
        subprocess.run(['kaggle', 'datasets', 'download', '-d',
                        'tommyngx/fluent-speech-corpus',
                        '-p', '/content/', '--unzip'], check=True)
        if os.path.exists('/content/fluent-speech-corpus'):
            os.rename('/content/fluent-speech-corpus', FSC_DIR)
        data_dir = os.path.join(FSC_DIR, 'data')
        for old_name, new_name in [('traindata.csv', 'train_data.csv'), ('validdata.csv', 'valid_data.csv')]:
            old_path = os.path.join(data_dir, old_name)
            new_path = os.path.join(data_dir, new_name)
            if os.path.exists(old_path) and not os.path.exists(new_path):
                os.rename(old_path, new_path)
        print('Downloaded from Kaggle!')
    except Exception as e:
        print(f'Kaggle failed ({e}), trying Zenodo...')
        subprocess.run(['wget', '-q',
                        'https://zenodo.org/records/3509828/files/fluent_speech_commands_dataset.tar.gz',
                        '-O', '/content/fsc.tar.gz'], check=True)
        subprocess.run(['tar', 'xzf', '/content/fsc.tar.gz', '-C', '/content/'], check=True)
        os.remove('/content/fsc.tar.gz')
        print('Downloaded from Zenodo!')
else:
    print(f'FSC already at {FSC_DIR}')

import sys, importlib.util
_nm_path = os.path.join(REPO, 'nanomamba.py')
assert os.path.isfile(_nm_path)
_spec = importlib.util.spec_from_file_location('nanomamba', _nm_path)
_nanomamba = importlib.util.module_from_spec(_spec)
sys.modules['nanomamba'] = _nanomamba
_spec.loader.exec_module(_nanomamba)
from nanomamba import create_nc_tcn_20k
print(f'Setup complete! NC-TCN: {sum(p.numel() for p in create_nc_tcn_20k().parameters()):,} params')

In [ ]:
# ==============================================================================
# Cell 1: Common Functions + Data Loading
# ==============================================================================
import os, sys, time, copy, random, json
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf

# Ensure nanomamba is importable
import importlib.util
REPO = '/content/NC-KWS-FineTuning'
if 'nanomamba' not in sys.modules:
    _nm_path = os.path.join(REPO, 'nanomamba.py')
    _spec = importlib.util.spec_from_file_location('nanomamba', _nm_path)
    _nanomamba = importlib.util.module_from_spec(_spec)
    sys.modules['nanomamba'] = _nanomamba
    _spec.loader.exec_module(_nanomamba)
from nanomamba import create_nc_tcn_20k, create_nc_tcn_20k_bi, create_nanomamba_nc_20k, create_nanomamba_nc_20k_bi

SR = 16000
MAX_LEN = SR * 3
FSC_DIR = '/content/fluent_speech_commands_dataset'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Load FSC ----
def load_fsc_split(split):
    csv_path = os.path.join(FSC_DIR, 'data', f'{split}_data.csv')
    df = pd.read_csv(csv_path)
    data = []
    for _, row in df.iterrows():
        intent = f"{row['action']}_{row['object']}_{row['location']}"
        wav_path = os.path.join(FSC_DIR, row['path'])
        data.append({'intent': intent, 'path': wav_path, 'text': row['transcription']})
    return data

def load_audio(path):
    a, sr = sf.read(path, dtype='float32')
    if a.ndim > 1: a = a[:, 0]
    if sr != SR:
        ratio = SR / sr
        n_out = int(len(a) * ratio)
        indices = np.linspace(0, len(a)-1, n_out).astype(np.float32)
        idx_f = np.floor(indices).astype(int)
        idx_c = np.minimum(idx_f + 1, len(a)-1)
        a = a[idx_f] * (1 - (indices - idx_f)) + a[idx_c] * (indices - idx_f)
    if len(a) > MAX_LEN: a = a[:MAX_LEN]
    elif len(a) < MAX_LEN: a = np.pad(a, (0, MAX_LEN - len(a)))
    return a.astype(np.float32)

print('Loading FSC data...')
train_data = load_fsc_split('train')
val_data = load_fsc_split('valid')
test_data = load_fsc_split('test')
all_intents = sorted(set(d['intent'] for d in train_data))
print(f'Total unique intents: {len(all_intents)}')

# ---- Intent split (same seed=42 as original) ----
random.seed(42)
intents_shuffled = all_intents.copy()
random.shuffle(intents_shuffled)
BASE_INTENTS = intents_shuffled[:25]
NEW_INTENTS = intents_shuffled[25:]
print(f'Base: {len(BASE_INTENTS)} | New: {len(NEW_INTENTS)}: {NEW_INTENTS}')

# ---- Load all audio ----
def build_dataset(data_list, intents, max_per_intent=None):
    result = defaultdict(list)
    for d in data_list:
        if d['intent'] in intents:
            if max_per_intent and len(result[d['intent']]) >= max_per_intent:
                continue
            try: result[d['intent']].append(load_audio(d['path']))
            except: continue
    return dict(result)

t0 = time.time()
base_train = build_dataset(train_data, BASE_INTENTS)
base_val = build_dataset(val_data, BASE_INTENTS)
base_test = build_dataset(test_data, BASE_INTENTS)
new_all_train = build_dataset(train_data, NEW_INTENTS)
new_test = build_dataset(test_data, NEW_INTENTS)
all_test_data = {**base_test, **new_test}
print(f'Audio loaded in {time.time()-t0:.1f}s')

# ---- Augmentation ----
def aug_time_stretch(a, rate):
    n = int(len(a) / rate)
    indices = np.linspace(0, len(a)-1, n).astype(np.float32)
    idx_floor = np.floor(indices).astype(int)
    idx_ceil = np.minimum(idx_floor + 1, len(a)-1)
    frac = indices - idx_floor
    stretched = a[idx_floor] * (1 - frac) + a[idx_ceil] * frac
    if len(stretched) > MAX_LEN: stretched = stretched[:MAX_LEN]
    elif len(stretched) < MAX_LEN: stretched = np.pad(stretched, (0, MAX_LEN - len(stretched)))
    return stretched.astype(np.float32)

def augment(a):
    out = a.copy()
    if np.random.random() < 0.5: out = aug_time_stretch(out, np.random.uniform(0.9, 1.1))
    if np.random.random() < 0.5: out = np.roll(out, np.random.randint(-4800, 4800))
    if np.random.random() < 0.5: out = out * (10 ** (np.random.uniform(-4, 4) / 20))
    if np.random.random() < 0.3:
        ml = int(len(out) * np.random.uniform(0.05, 0.1))
        s = np.random.randint(0, len(out) - ml)
        out[s:s+ml] = 0.0
    out = out + np.random.randn(len(out)).astype(np.float32) * 0.003
    return out.astype(np.float32)

# ---- LoRA ----
class LoRALinear(nn.Module):
    def __init__(self, original, rank=2, alpha=4):
        super().__init__()
        self.original = original
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(original.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original.out_features))
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling

def label_smooth_ce(logits, targets, smoothing=0.1):
    log_probs = F.log_softmax(logits, dim=-1)
    nll = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
    smooth = -log_probs.mean(dim=-1)
    return ((1 - smoothing) * nll + smoothing * smooth).mean()

# ---- Core Functions ----
def train_base_slu(intents, train_data, val_data, epochs=30, model_fn=create_nc_tcn_20k):
    n_cls = len(intents)
    model = model_fn(n_classes=n_cls).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Base model: {n_cls} intents, {n_params:,} params, {epochs} epochs')
    X, Y = [], []
    for i, intent in enumerate(intents):
        for a in train_data.get(intent, []):
            X.append(a); Y.append(i)
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(X))
        eloss, nb = 0, 0
        for i in range(0, len(X), 32):
            bi = perm[i:i+32]
            batch = [augment(X[j]) if np.random.random() < 0.5 else X[j] for j in bi]
            bx = torch.stack([torch.from_numpy(a).float() for a in batch]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = F.cross_entropy(model(bx), by)
            opt.zero_grad(); loss.backward(); opt.step()
            eloss += loss.item(); nb += 1
        sched.step()
        if (ep+1) % 10 == 0 or ep == 0:
            model.eval(); vc, vt = 0, 0
            with torch.no_grad():
                for i, intent in enumerate(intents):
                    for a in val_data.get(intent, []):
                        x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                        pred = model(x).argmax(-1).item()
                        vt += 1; vc += (pred == i)
            print(f'    Ep {ep+1}/{epochs} loss={eloss/nb:.4f} val={vc/max(vt,1)*100:.1f}%')
    return model

def evaluate_slu(model, intents, test_data):
    model.eval()
    correct, total = 0, 0
    per_intent = {}
    with torch.no_grad():
        for i, intent in enumerate(intents):
            ic, it = 0, 0
            for a in test_data.get(intent, []):
                x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                pred = model(x).argmax(-1).item()
                it += 1; ic += (pred == i)
            per_intent[intent] = ic / max(it, 1)
            correct += ic; total += it
    return correct / max(total, 1), per_intent

def ncopal_slu(base_model, base_intents, new_intents, new_train, base_train_sub,
               lambda_kd=5.0, kd_temp=3.0, lora_rank=2, lora_alpha=4,
               stage1_ep=5, stage2_ep=20, rehearsal=30, aug_new=8, aug_old=4,
               stage1_lr=4.5e-3, stage2_lr=1.5e-3):
    """NC-OPAL with configurable hyperparameters for sweep experiments."""
    model = copy.deepcopy(base_model)
    n_old = len(base_intents)
    n_new = len(new_intents)
    n_total = n_old + n_new
    d = model.classifier.in_features

    emb_hook = {}
    def hook_fn(m, inp, out): emb_hook['e'] = inp[0].detach()
    def get_emb(audio):
        h = model.classifier.register_forward_hook(hook_fn)
        with torch.no_grad(): model(torch.from_numpy(audio).float().unsqueeze(0).to(DEVICE))
        h.remove()
        return emb_hook['e'].squeeze(0)

    old_head = model.classifier
    new_head = nn.Linear(d, n_total).to(DEVICE)
    with torch.no_grad():
        new_head.weight[:n_old] = old_head.weight
        new_head.bias[:n_old] = old_head.bias
        scale = old_head.weight.norm(dim=1).mean().item()
        for i, intent in enumerate(new_intents):
            embs = [get_emb(a) for a in new_train.get(intent, [])]
            if embs:
                proto = F.normalize(torch.stack(embs).mean(0), dim=0)
                new_head.weight[n_old+i] = proto * scale
                new_head.bias[n_old+i] = old_head.bias.mean().item()
    model.classifier = new_head

    loras = []
    if lora_rank >= 1:
        for block in model.blocks:
            for attr in ['in_proj', 'out_proj']:
                if hasattr(block, attr):
                    orig = getattr(block, attr)
                    if isinstance(orig, nn.Linear):
                        lo = LoRALinear(orig, rank=lora_rank, alpha=lora_alpha).to(DEVICE)
                        setattr(block, attr, lo); loras.append(lo)

    teacher = copy.deepcopy(base_model).to(DEVICE)
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False

    X, Y = [], []
    for i, intent in enumerate(new_intents):
        li = n_old + i
        for a in new_train.get(intent, []):
            X.append(a); Y.append(li)
            for _ in range(aug_new): X.append(augment(a)); Y.append(li)
    for i, intent in enumerate(base_intents):
        for a in base_train_sub.get(intent, [])[:rehearsal]:
            X.append(a); Y.append(i)
            for _ in range(aug_old): X.append(augment(a)); Y.append(i)

    # Stage 1: Head only
    for p in model.parameters(): p.requires_grad = False
    for p in new_head.parameters(): p.requires_grad = True
    opt1 = torch.optim.AdamW(new_head.parameters(), lr=stage1_lr, weight_decay=0.01)
    model.train()
    for ep in range(stage1_ep):
        perm = np.random.permutation(len(X))
        for i in range(0, len(X), 32):
            bi = perm[i:i+32]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = label_smooth_ce(model(bx), by, 0.1)
            opt1.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(new_head.parameters(), 1.0)
            opt1.step()

    # Stage 2: LoRA + Head + KD
    if stage2_ep <= 0:
        return model, base_intents + new_intents
    for m in loras:
        for p in m.parameters(): p.requires_grad = True
    param_groups = []
    if loras:
        param_groups.append({'params': [p for m in loras for p in m.parameters()], 'lr': stage2_lr})
    param_groups.append({'params': new_head.parameters(), 'lr': stage2_lr * 0.5})
    opt2 = torch.optim.AdamW(param_groups, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=stage2_ep)

    for ep in range(stage2_ep):
        perm = np.random.permutation(len(X))
        ecl, ekd, nbt = 0, 0, 0
        for i in range(0, len(X), 32):
            bi = perm[i:i+32]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            logits = model(bx)
            cls_loss = label_smooth_ce(logits, by, 0.05)
            kd_loss = torch.tensor(0.0).to(DEVICE)
            old_mask = by < n_old
            if old_mask.any():
                with torch.no_grad(): t_logits = teacher(bx[old_mask])
                s_base = logits[old_mask][:, :n_old] / kd_temp
                t_base = t_logits / kd_temp
                kd_loss = F.kl_div(F.log_softmax(s_base, -1), F.softmax(t_base, -1),
                                   reduction='batchmean') * (kd_temp**2)
            kd_w = lambda_kd * min(1.0, ep / 5.0)
            total = cls_loss + kd_w * kd_loss
            opt2.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt2.step()
            ecl += cls_loss.item(); ekd += kd_loss.item(); nbt += 1
        sched.step()
    return model, base_intents + new_intents

def run_eval(model, intents, base_intents, new_intents, test_data, base_ref_acc):
    """Evaluate and return dict of results."""
    acc, per = evaluate_slu(model, intents, test_data)
    base_acc = np.mean([per[i] for i in base_intents])
    new_acc = np.mean([per[i] for i in new_intents])
    forget = (base_ref_acc - base_acc) * 100
    return {'overall': acc, 'base': base_acc, 'new': new_acc, 'forget': forget, 'per': per}

print(f'Functions loaded. Device: {DEVICE}')


In [ ]:
# ==============================================================================
# Cell 2: EXPERIMENT 1 - Base 50 Epoch (4 Backbones)
# Goal: Check if SSM/SSM-Bi improve with longer base training
# Estimated time: ~35 min on T4
# ==============================================================================
print('='*70)
print('  EXPERIMENT 1: Base 50 Epoch - All 4 Backbones')
print('='*70)

N_SHOT = 20
new_train = {k: v[:N_SHOT] for k, v in new_all_train.items()}
base_train_sub = {k: v[:15] for k, v in base_train.items()}

BACKBONES_EXP1 = [
    ('NC-TCN',     create_nc_tcn_20k,          'Causal TCN'),
    ('NC-TCN-Bi',  create_nc_tcn_20k_bi,       'Bidirectional TCN'),
    ('NC-SSM',     create_nanomamba_nc_20k,     'Selective SSM'),
    ('NC-SSM-Bi',  create_nanomamba_nc_20k_bi,  'Bidirectional SSM'),
]

exp1_results = {}
_model_dir = os.path.join(REPO, 'results', 'models')
os.makedirs(_model_dir, exist_ok=True)

for bb_name, bb_fn, bb_desc in BACKBONES_EXP1:
    print(f'\n{"="*60}')
    print(f'  {bb_name} ({bb_desc}) - 50 epochs base')
    print(f'{"="*60}')
    t0 = time.time()

    # Base training (50 epochs)
    base_model = train_base_slu(BASE_INTENTS, base_train, base_val, epochs=50, model_fn=bb_fn)
    base_acc, base_per = evaluate_slu(base_model, BASE_INTENTS, base_test)
    torch.save(base_model.state_dict(), os.path.join(_model_dir, f'{bb_name}_base_50ep.pt'))
    t_base = time.time() - t0
    print(f'  Base acc: {base_acc*100:.1f}% [{t_base:.0f}s]')

    # NC-OPAL (default settings)
    t1 = time.time()
    opal_model, opal_intents = ncopal_slu(base_model, BASE_INTENTS, NEW_INTENTS, new_train, base_train_sub)
    torch.save(opal_model.state_dict(), os.path.join(_model_dir, f'{bb_name}_ncopal_50ep.pt'))
    r = run_eval(opal_model, opal_intents, BASE_INTENTS, NEW_INTENTS, all_test_data, base_acc)
    t_opal = time.time() - t1
    r['base_ref'] = base_acc
    r['time'] = time.time() - t0
    n_params = sum(p.numel() for p in bb_fn(n_classes=31).parameters())
    r['params'] = n_params
    exp1_results[bb_name] = r
    print(f'  NC-OPAL: All={r["overall"]*100:.1f}% Base={r["base"]*100:.1f}% New={r["new"]*100:.1f}% Fgt={r["forget"]:+.1f}% [{t_opal:.0f}s]')

# Summary
print(f'\n{"="*70}')
print(f'{"Backbone":<14} {"Params":>7} {"All":>7} {"Base":>7} {"New":>7} {"Fgt":>7} {"Time":>6}')
print('-'*60)
for bb, r in exp1_results.items():
    print(f'{bb:<14} {r["params"]:>6,} {r["overall"]*100:>6.1f}% {r["base"]*100:>6.1f}% {r["new"]*100:>6.1f}% {r["forget"]:>+6.1f}% {r["time"]:>5.0f}s')

# Compare with 30ep results
prev = {'NC-TCN': 86.3, 'NC-TCN-Bi': 84.8, 'NC-SSM': 79.0, 'NC-SSM-Bi': 85.0}
print(f'\n--- vs 30ep ---')
for bb, r in exp1_results.items():
    diff = r['overall']*100 - prev.get(bb, 0)
    print(f'  {bb}: {prev.get(bb,0):.1f}% -> {r["overall"]*100:.1f}% ({diff:+.1f}%p)')

json.dump(exp1_results, open(os.path.join(REPO, 'results', 'exp1_50ep.json'), 'w'), indent=2, default=str)
print('\nExp1 saved!')

In [ ]:
# ==============================================================================
# Cell 3: EXPERIMENT 2 - N_SHOT Sweep (NC-TCN, best backbone)
# Goal: Few-shot scaling curve -> Figure for paper
# Estimated time: ~25 min on T4
# ==============================================================================
print('='*70)
print('  EXPERIMENT 2: Few-Shot Scaling (K = 5, 10, 20, 30, 50)')
print('  Backbone: NC-TCN (best overall)')
print('='*70)

# Train base model once (50ep from Exp1, or retrain)
base_train_sub = {k: v[:15] for k, v in base_train.items()}
_model_dir = os.path.join(REPO, 'results', 'models')
os.makedirs(_model_dir, exist_ok=True)
print('\nTraining NC-TCN base (50ep)...')
tcn_base = train_base_slu(BASE_INTENTS, base_train, base_val, epochs=50, model_fn=create_nc_tcn_20k)
tcn_base_acc, _ = evaluate_slu(tcn_base, BASE_INTENTS, base_test)
torch.save(tcn_base.state_dict(), os.path.join(_model_dir, 'NC-TCN_base_50ep_nshot.pt'))
print(f'  Base acc: {tcn_base_acc*100:.1f}%')

SHOTS = [5, 10, 20, 30, 50]
exp2_results = {}

for K in SHOTS:
    print(f'\n--- K={K} ({K}-shot) ---')
    t0 = time.time()
    new_train_k = {k: v[:K] for k, v in new_all_train.items()}
    actual_k = min(len(v) for v in new_train_k.values())
    print(f'  Actual shots per intent: {actual_k}')

    opal_model, opal_intents = ncopal_slu(
        tcn_base, BASE_INTENTS, NEW_INTENTS, new_train_k, base_train_sub)
    torch.save(opal_model.state_dict(), os.path.join(_model_dir, f'NC-TCN_ncopal_K{K}.pt'))
    r = run_eval(opal_model, opal_intents, BASE_INTENTS, NEW_INTENTS, all_test_data, tcn_base_acc)
    r['K'] = K
    r['time'] = time.time() - t0
    exp2_results[K] = r
    print(f'  All={r["overall"]*100:.1f}% Base={r["base"]*100:.1f}% New={r["new"]*100:.1f}% Fgt={r["forget"]:+.1f}% [{r["time"]:.0f}s]')

    # Per-intent for confusable pair analysis
    print(f'  Per-intent (new):')
    for intent in NEW_INTENTS:
        print(f'    {intent}: {r["per"].get(intent,0)*100:.1f}%')

print(f'\n{"="*70}')
print(f'{"K":>4} {"All":>7} {"Base":>7} {"New":>7} {"Fgt":>7}')
print('-'*40)
for K, r in exp2_results.items():
    print(f'{K:>4} {r["overall"]*100:>6.1f}% {r["base"]*100:>6.1f}% {r["new"]*100:>6.1f}% {r["forget"]:>+6.1f}%')

json.dump(exp2_results, open(os.path.join(REPO, 'results', 'exp2_nshot.json'), 'w'), indent=2, default=str)
print('\nExp2 saved!')


In [ ]:
# ==============================================================================
# Cell 4: EXPERIMENT 3 - SLU Ablation Study (NC-TCN)
# Goal: Direct SLU ablation -> Replace indirect KWS ablation in paper
# Estimated time: ~15 min on T4
# ==============================================================================
print('='*70)
print('  EXPERIMENT 3: SLU Ablation Study (NC-TCN)')
print('  Configs: Head-only | +LoRA (no KD) | +KD (no LoRA) | Full NC-OPAL')
print('='*70)

# Reuse base model from Exp2 or retrain
N_SHOT = 20
_model_dir = os.path.join(REPO, 'results', 'models')
os.makedirs(_model_dir, exist_ok=True)
new_train = {k: v[:N_SHOT] for k, v in new_all_train.items()}
base_train_sub = {k: v[:15] for k, v in base_train.items()}

base_train_sub = {k: v[:15] for k, v in base_train.items()}
try:
    tcn_base  # from Exp2
    print('Using TCN base from Exp2')
except:
    print('Training TCN base (50ep)...')
    tcn_base = train_base_slu(BASE_INTENTS, base_train, base_val, epochs=50, model_fn=create_nc_tcn_20k)
    tcn_base_acc, _ = evaluate_slu(tcn_base, BASE_INTENTS, base_test)

ablation_configs = [
    ('Head-only (S1)',     dict(stage1_ep=5,  stage2_ep=0,  lambda_kd=0,  lora_rank=0)),
    ('+LoRA (no KD)',      dict(stage1_ep=5,  stage2_ep=20, lambda_kd=0,  lora_rank=2)),
    ('+KD (no LoRA)',      dict(stage1_ep=5,  stage2_ep=20, lambda_kd=5,  lora_rank=0)),
    ('Full NC-OPAL',      dict(stage1_ep=5,  stage2_ep=20, lambda_kd=5,  lora_rank=2)),
    ('NC-OPAL r=4',       dict(stage1_ep=5,  stage2_ep=20, lambda_kd=5,  lora_rank=4, lora_alpha=8)),
    ('NC-OPAL kd=10',     dict(stage1_ep=5,  stage2_ep=20, lambda_kd=10, lora_rank=2)),
]

exp3_results = {}

for name, cfg in ablation_configs:
    print(f'\n--- {name} ---')
    t0 = time.time()

    opal_model, opal_intents = ncopal_slu(
        tcn_base, BASE_INTENTS, NEW_INTENTS, new_train, base_train_sub,
        stage1_ep=cfg.get('stage1_ep', 5),
        stage2_ep=cfg.get('stage2_ep', 20),
        lambda_kd=cfg.get('lambda_kd', 5.0),
        lora_rank=cfg.get('lora_rank', 2),
        lora_alpha=cfg.get('lora_alpha', 4))

    torch.save(opal_model.state_dict(), os.path.join(_model_dir, f'ablation_{name.replace(" ","_")}.pt'))
    r = run_eval(opal_model, opal_intents, BASE_INTENTS, NEW_INTENTS, all_test_data, tcn_base_acc)
    r['time'] = time.time() - t0
    r['config'] = cfg
    exp3_results[name] = r
    print(f'  All={r["overall"]*100:.1f}% Base={r["base"]*100:.1f}% New={r["new"]*100:.1f}% Fgt={r["forget"]:+.1f}% [{r["time"]:.0f}s]')

print(f'\n{"="*70}')
print(f'{"Config":<20} {"All":>7} {"Base":>7} {"New":>7} {"Fgt":>7}')
print('-'*55)
for name, r in exp3_results.items():
    print(f'{name:<20} {r["overall"]*100:>6.1f}% {r["base"]*100:>6.1f}% {r["new"]*100:>6.1f}% {r["forget"]:>+6.1f}%')

json.dump(exp3_results, open(os.path.join(REPO, 'results', 'exp3_ablation.json'), 'w'), indent=2, default=str)
print('\nExp3 saved!')


In [ ]:
# ==============================================================================
# Cell 5: EXPERIMENT 4 - Stage2 Epoch + Hyperparameter Sweep
# Goal: Find optimal config for maximum accuracy
# Estimated time: ~20 min on T4
# ==============================================================================
print('='*70)
print('  EXPERIMENT 4: Hyperparameter Optimization')
print('  Stage2 epochs + rehearsal + augmentation sweep')
print('='*70)

N_SHOT = 20
_model_dir = os.path.join(REPO, 'results', 'models')
os.makedirs(_model_dir, exist_ok=True)
new_train = {k: v[:N_SHOT] for k, v in new_all_train.items()}

try:
    tcn_base
except:
    tcn_base = train_base_slu(BASE_INTENTS, base_train, base_val, epochs=50, model_fn=create_nc_tcn_20k)
    tcn_base_acc, _ = evaluate_slu(tcn_base, BASE_INTENTS, base_test)

hp_configs = [
    # (name, {ncopal kwargs})
    ('S2=20 (baseline)',   dict(stage2_ep=20, rehearsal=30, aug_new=8, aug_old=4, lambda_kd=5)),
    ('S2=30',              dict(stage2_ep=30, rehearsal=30, aug_new=8, aug_old=4, lambda_kd=5)),
    ('S2=40',              dict(stage2_ep=40, rehearsal=30, aug_new=8, aug_old=4, lambda_kd=5)),
    ('reh=50',             dict(stage2_ep=20, rehearsal=50, aug_new=8, aug_old=4, lambda_kd=5)),
    ('aug12x',             dict(stage2_ep=20, rehearsal=30, aug_new=12, aug_old=6, lambda_kd=5)),
    ('S2=30+reh50+aug12',  dict(stage2_ep=30, rehearsal=50, aug_new=12, aug_old=6, lambda_kd=5)),
    ('kd=7+S2=30',        dict(stage2_ep=30, rehearsal=30, aug_new=8, aug_old=4, lambda_kd=7)),
    ('BEST (S2=30+r50+a12+kd7)', dict(stage2_ep=30, rehearsal=50, aug_new=12, aug_old=6, lambda_kd=7)),
]

exp4_results = {}

for name, cfg in hp_configs:
    print(f'\n--- {name} ---')
    t0 = time.time()
    opal_model, opal_intents = ncopal_slu(
        tcn_base, BASE_INTENTS, NEW_INTENTS, new_train, base_train, **cfg)
    torch.save(opal_model.state_dict(), os.path.join(_model_dir, f'hp_{name.replace(" ","_").replace("/","_")}.pt'))
    r = run_eval(opal_model, opal_intents, BASE_INTENTS, NEW_INTENTS, all_test_data, tcn_base_acc)
    r['time'] = time.time() - t0
    r['config'] = cfg
    exp4_results[name] = r
    print(f'  All={r["overall"]*100:.1f}% Base={r["base"]*100:.1f}% New={r["new"]*100:.1f}% Fgt={r["forget"]:+.1f}% [{r["time"]:.0f}s]')

    # Per-intent for decrease_heat analysis
    for intent in NEW_INTENTS:
        print(f'    {intent}: {r["per"].get(intent,0)*100:.1f}%')

print(f'\n{"="*70}')
print(f'{"Config":<30} {"All":>7} {"Base":>7} {"New":>7} {"Fgt":>7}')
print('-'*65)
for name, r in exp4_results.items():
    print(f'{name:<30} {r["overall"]*100:>6.1f}% {r["base"]*100:>6.1f}% {r["new"]*100:>6.1f}% {r["forget"]:>+6.1f}%')

# Find best
best = max(exp4_results.items(), key=lambda x: x[1]['overall'])
print(f'\n*** BEST: {best[0]} -> {best[1]["overall"]*100:.1f}% ***')

json.dump(exp4_results, open(os.path.join(REPO, 'results', 'exp4_hpsweep.json'), 'w'), indent=2, default=str)
print('\nExp4 saved!')


In [ ]:
# ==============================================================================
# Cell 6: Summary + Visualization
# ==============================================================================
import matplotlib.pyplot as plt

print('='*70)
print('  FULL EXPERIMENT SUMMARY')
print('='*70)

# ---- Exp1: 50ep vs 30ep comparison ----
print('\n[Exp1] Base 50 Epoch Results:')
prev_30ep = {'NC-TCN': (86.3, 76.8, -5.3), 'NC-TCN-Bi': (84.8, 81.5, -6.8),
             'NC-SSM': (79.0, 83.2, -9.3), 'NC-SSM-Bi': (85.0, 75.9, -4.9)}
for bb, r in exp1_results.items():
    old = prev_30ep.get(bb, (0,0,0))
    print(f'  {bb}: All {old[0]:.1f}%->{r["overall"]*100:.1f}% | New {old[1]:.1f}%->{r["new"]*100:.1f}% | Fgt {old[2]:+.1f}%->{r["forget"]:+.1f}%')

# ---- Exp2: N_SHOT scaling curve plot ----
print('\n[Exp2] Few-Shot Scaling Curve:')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ks = sorted(exp2_results.keys())
alls = [exp2_results[k]['overall']*100 for k in ks]
news = [exp2_results[k]['new']*100 for k in ks]
fgts = [exp2_results[k]['forget'] for k in ks]

ax1.plot(ks, alls, 'bo-', label='Overall', linewidth=2, markersize=8)
ax1.plot(ks, news, 'gs-', label='New Intent', linewidth=2, markersize=8)
ax1.set_xlabel('K (shots per intent)', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('NC-OPAL Few-Shot Scaling (NC-TCN)', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(ks)
for k, a, n in zip(ks, alls, news):
    ax1.annotate(f'{a:.1f}', (k, a), textcoords='offset points', xytext=(0,8), ha='center', fontsize=9)
    ax1.annotate(f'{n:.1f}', (k, n), textcoords='offset points', xytext=(0,-12), ha='center', fontsize=9)

ax2.plot(ks, fgts, 'r^-', label='Forgetting', linewidth=2, markersize=8)
ax2.set_xlabel('K (shots per intent)', fontsize=12)
ax2.set_ylabel('Forgetting (%)', fontsize=12)
ax2.set_title('Forgetting vs K', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(ks)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(REPO, 'results', 'nshot_scaling.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Exp3: Ablation bar chart ----
print('\n[Exp3] SLU Ablation:')
fig, ax = plt.subplots(figsize=(10, 5))
names = list(exp3_results.keys())
overall = [exp3_results[n]['overall']*100 for n in names]
new_acc = [exp3_results[n]['new']*100 for n in names]
x = np.arange(len(names))
w = 0.35
ax.bar(x - w/2, overall, w, label='Overall', color='#4285F4')
ax.bar(x + w/2, new_acc, w, label='New Intent', color='#34A853')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Accuracy (%)')
ax.set_title('NC-OPAL Ablation Study (NC-TCN, 20-shot)')
ax.legend()
ax.grid(True, alpha=0.2, axis='y')
for i, (o, n) in enumerate(zip(overall, new_acc)):
    ax.text(i-w/2, o+0.5, f'{o:.1f}', ha='center', fontsize=8)
    ax.text(i+w/2, n+0.5, f'{n:.1f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(REPO, 'results', 'ablation_slu.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Exp4: Best config ----
print('\n[Exp4] HP Sweep Best:')
best = max(exp4_results.items(), key=lambda x: x[1]['overall'])
print(f'  {best[0]}: All={best[1]["overall"]*100:.1f}% New={best[1]["new"]*100:.1f}% Fgt={best[1]["forget"]:+.1f}%')

# ---- FINAL BEST ----
all_results = {}
for d in [exp1_results, exp2_results, exp3_results, exp4_results]:
    for k, v in d.items():
        all_results[str(k)] = v
best_overall = max(all_results.items(), key=lambda x: x[1]['overall'])
print(f'\n*** OVERALL BEST: {best_overall[0]} -> {best_overall[1]["overall"]*100:.1f}% ***')

# ---- Hard Intent Analysis ----
print('\n[Hard Intent Analysis]')
print('Confusable pairs (NEW vs BASE):')
ACTION_PAIRS = {'decrease': 'increase', 'increase': 'decrease',
                'activate': 'deactivate', 'deactivate': 'activate'}
HARD_INTENTS, EASY_INTENTS_D = [], []
for ni in NEW_INTENTS:
    parts = ni.split('_')
    action = parts[0]
    rest = '_'.join(parts[1:])
    counterpart_action = ACTION_PAIRS.get(action)
    if counterpart_action:
        counterpart = f'{counterpart_action}_{rest}'
        if counterpart in BASE_INTENTS:
            HARD_INTENTS.append(ni)
            print(f'  HARD: {ni} <-> {counterpart}')
            continue
    EASY_INTENTS_D.append(ni)
    print(f'  EASY: {ni}')
print(f'Hard: {len(HARD_INTENTS)}, Easy: {len(EASY_INTENTS_D)}')

if exp2_results and HARD_INTENTS:
    print(f'\n{"K":>4} {"Hard":>9} {"Easy":>9} {"Gap":>7}')
    print('-'*35)
    for K in sorted(exp2_results.keys()):
        r = exp2_results[K]
        hard_acc = np.mean([r['per'].get(i, 0) for i in HARD_INTENTS])
        easy_acc = np.mean([r['per'].get(i, 0) for i in EASY_INTENTS_D]) if EASY_INTENTS_D else 0
        gap = (easy_acc - hard_acc) * 100
        print(f'{K:>4} {hard_acc*100:>8.1f}% {easy_acc*100:>8.1f}% {gap:>+6.1f}%p')

# Commercialization readiness
print('\n[Commercialization Readiness]')
best_all = max(all_results.items(), key=lambda x: x[1]['overall'])
best_acc = best_all[1]['overall'] * 100
targets = [('Wearable MVP', 85), ('Smart Home', 90), ('Industrial', 95)]
for name, target in targets:
    status = '[OK]' if best_acc >= target else '[--]'
    gap = best_acc - target
    print(f'  {status} {name}: {target}% (gap: {gap:+.1f}%p)')


# Push to GitHub
import subprocess
subprocess.run(['git', '-C', REPO, 'add', 'results/'], check=True)
subprocess.run(['git', '-C', REPO, 'commit', '-m', 'Experiments: 50ep + nshot + ablation + hpsweep'], check=False)
subprocess.run(['git', '-C', REPO, 'push'], check=False)
print('All results pushed to GitHub!')
